<a href="https://colab.research.google.com/github/LINWOO0099/machine-learning/blob/main/Supervised_Machine_Learning_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [26]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression, Ridge, LogisticRegression
from sklearn.metrics import (
    mean_squared_error, r2_score,
    confusion_matrix, classification_report,
    roc_curve, roc_auc_score,
    precision_score, recall_score, f1_score
)
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.impute import SimpleImputer # Added import
from sklearn.pipeline import Pipeline # Added import
import warnings
warnings.filterwarnings('ignore')

# ------------------------------------------------------------
# 1. Load data and define targets
# ------------------------------------------------------------
df = pd.read_csv('cleaned_data.csv')
print(df.head())
TARGET_REG = 'Price'          # <-- change to your column
y_reg = df[TARGET_REG]
y_clf = (y_reg > y_reg.median()).astype(int)
X = df.drop(columns=[TARGET_REG])
del df   # free original DataFrame

# ------------------------------------------------------------
# 2. Train/test split
# ------------------------------------------------------------
X_train, X_test, y_reg_train, y_reg_test, y_clf_train, y_clf_test = train_test_split(
    X, y_reg, y_clf, test_size=0.2, random_state=42
)
del X, y_reg, y_clf   # free full data



  Invoice StockCode                          Description  Quantity  \
0  489434     85048  15CM CHRISTMAS GLASS BALL 20 LIGHTS        12   
1  489434    79323P                   PINK CHERRY LIGHTS        12   
2  489434    79323W                  WHITE CHERRY LIGHTS        12   
3  489434     22041         RECORD FRAME 7" SINGLE SIZE         48   
4  489434     21232       STRAWBERRY CERAMIC TRINKET BOX        24   

           InvoiceDate  Price  Customer ID         Country  
0  2009-12-01 07:45:00   6.95      13085.0  United Kingdom  
1  2009-12-01 07:45:00   6.75      13085.0  United Kingdom  
2  2009-12-01 07:45:00   6.75      13085.0  United Kingdom  
3  2009-12-01 07:45:00   2.10      13085.0  United Kingdom  
4  2009-12-01 07:45:00   1.25      13085.0  United Kingdom  


In [12]:
# ------------------------------------------------------------
# 3. Encoding and Scaling Pipeline
# ------------------------------------------------------------
cat_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
num_cols = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()

# Ordinal mappings (if any)
ordinal_mappings = {
    # 'education': ['High School', 'Bachelor', 'Master', 'PhD'],
}

# Define preprocessing pipelines for numerical and categorical features
numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(drop='first',
                             handle_unknown='ignore',
                             sparse_output=False, # Changed to False for dense output
                             min_frequency=0.01,
                             max_categories=20))
])

# Collect all transformers for ColumnTransformer
preprocessor_steps = []

# Handle ordinal columns if defined
for col in cat_cols:
    if col in ordinal_mappings:
        preprocessor_steps.append(
            (f'ordinal_{col}',
             Pipeline(steps=[
                 ('imputer', SimpleImputer(strategy='constant', fill_value='missing_category')),
                 ('ordinal', OrdinalEncoder(categories=[ordinal_mappings[col]],
                                            handle_unknown='use_encoded_value',
                                            unknown_value=-1))
             ]),
             [col])
        )
    elif col in X_train.columns: # Ensure the column actually exists in X_train
        preprocessor_steps.append((f'onehot_{col}', categorical_transformer, [col]))

# Add numerical transformer if there are numerical columns
if num_cols:
    preprocessor_steps.append(('num', numerical_transformer, num_cols))

preprocessor = ColumnTransformer(preprocessor_steps, remainder='drop') # Drop any unhandled columns

# Wrap the ColumnTransformer in a final Pipeline with an imputer to catch any lingering NaNs
full_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('final_imputer', SimpleImputer(strategy='mean')) # Safety net for any remaining NaNs
])

# Fit and transform
X_train_scaled = full_pipeline.fit_transform(X_train).astype(np.float32)
X_test_scaled = full_pipeline.transform(X_test).astype(np.float32)

# Debugging: Check for NaNs
print(f"NaNs in X_train_scaled after full pipeline: {np.isnan(X_train_scaled).sum()}")
print(f"NaNs in X_test_scaled after full pipeline: {np.isnan(X_test_scaled).sum()}")

# Feature names generation using preprocessor.get_feature_names_out()
feature_names = full_pipeline.named_steps['preprocessor'].get_feature_names_out()


NaNs in X_train_scaled after full pipeline: 0
NaNs in X_test_scaled after full pipeline: 0


In [18]:
# ------------------------------------------------------------
# 4. Scaling – fit only on training, transform both (now handled in step 3)
# ------------------------------------------------------------
# The scaling and imputation are now performed within the ColumnTransformer and full_pipeline in step 3.
# X_train_scaled and X_test_scaled are already ready for model training.

# ------------------------------------------------------------
# 5. Regression models (unchanged)
# ------------------------------------------------------------
lr = LinearRegression()
lr.fit(X_train_scaled, y_reg_train)
y_pred_lr = lr.predict(X_test_scaled)
mse_lr = mean_squared_error(y_reg_test, y_pred_lr)
r2_lr = r2_score(y_reg_test, y_pred_lr)

ridge = Ridge(alpha=1.0)
ridge.fit(X_train_scaled, y_reg_train)
y_pred_ridge = ridge.predict(X_test_scaled)
mse_ridge = mean_squared_error(y_reg_test, y_pred_ridge)
r2_ridge = r2_score(y_reg_test, y_pred_ridge)

# Print or store results (as before)
print(f"Linear Regression MSE: {mse_lr:.4f}, R2: {r2_lr:.4f}")
print(f"Ridge Regression MSE: {mse_ridge:.4f}, R2: {r2_ridge:.4f}")

Linear Regression MSE: 7008.4205, R2: 0.0001
Ridge Regression MSE: 7008.4204, R2: 0.0001


In [19]:

# ------------------------------------------------------------
# 6. Classification – use class_weight instead of SMOTE if possible
# ------------------------------------------------------------
minority_ratio = y_clf_train.value_counts().min() / len(y_clf_train)

if minority_ratio < 0.20:   # only use SMOTE when severely imbalanced
    print("Applying SMOTE (with reduced sampling strategy).")
    smote = SMOTE(sampling_strategy=0.5, random_state=42)  # generate fewer samples
    lr_pipeline = ImbPipeline([
        ('smote', smote),
        ('classifier', LogisticRegression(max_iter=1000, random_state=42))
    ])
    lr_pipeline.fit(X_train_scaled, y_clf_train)
    lr_model = lr_pipeline.named_steps['classifier']
else:
    print("Using class_weight='balanced' (no data duplication).")
    lr_model = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
    lr_model.fit(X_train_scaled, y_clf_train)

# Predictions and probabilities
y_pred_clf = lr_model.predict(X_test_scaled)
y_proba = lr_model.predict_proba(X_test_scaled)[:, 1]

# Metrics (confusion matrix, classification report, ROC, AUC) – as before

Using class_weight='balanced' (no data duplication).


In [21]:

# ------------------------------------------------------------
# 7. Threshold sensitivity – unchanged
# ------------------------------------------------------------
thresholds = np.arange(0.30, 0.71, 0.10)
results = []
for th in thresholds:
    preds = (y_proba >= th).astype(int)
    results.append([th,
                    precision_score(y_clf_test, preds),
                    recall_score(y_clf_test, preds),
                    f1_score(y_clf_test, preds)])
# Print table
print("\nThreshold Sensitivity Analysis:")
print(f"{'Threshold':<10} | {'Precision':<10} | {'Recall':<10} | {'F1-Score':<10}")
print("-" * 50)
for row in results:
    print(f"{row[0]:<10.2f} | {row[1]:<10.4f} | {row[2]:<10.4f} | {row[3]:<10.4f}")


Threshold Sensitivity Analysis:
Threshold  | Precision  | Recall     | F1-Score  
--------------------------------------------------
0.30       | 0.4875     | 0.9954     | 0.6544    
0.40       | 0.4950     | 0.9871     | 0.6594    
0.50       | 0.5777     | 0.8515     | 0.6884    
0.60       | 0.1190     | 0.0008     | 0.0016    
0.70       | 0.0950     | 0.0003     | 0.0007    


In [23]:

# ------------------------------------------------------------
# 8. Regularization experiment (C=0.01)
# ------------------------------------------------------------
# Train second model (same choice as above)
if minority_ratio < 0.20:
    lr_c01_pipeline = ImbPipeline([
        ('smote', SMOTE(sampling_strategy=0.5, random_state=42)),
        ('classifier', LogisticRegression(C=0.01, max_iter=1000, random_state=42))
    ])
    lr_c01_pipeline.fit(X_train_scaled, y_clf_train)
    lr_c01 = lr_c01_pipeline.named_steps['classifier']
else:
    lr_c01 = LogisticRegression(C=0.01, max_iter=1000, class_weight='balanced', random_state=42)
    lr_c01.fit(X_train_scaled, y_clf_train)

y_proba_c01 = lr_c01.predict_proba(X_test_scaled)[:, 1]
y_pred_c01 = lr_c01.predict(X_test_scaled)

# Compute metrics and compare
print("\n--- Regularization Experiment (C=0.01) ---")

# Metrics for default LR model (from section 6)
auc_default = roc_auc_score(y_clf_test, y_proba)
precision_default = precision_score(y_clf_test, y_pred_clf)
recall_default = recall_score(y_clf_test, y_pred_clf)
f1_default = f1_score(y_clf_test, y_pred_clf)

print("\nDefault Logistic Regression (C=1.0):")
print(f"  ROC AUC: {auc_default:.4f}")
print(f"  Precision: {precision_default:.4f}")
print(f"  Recall: {recall_default:.4f}")
print(f"  F1-Score: {f1_default:.4f}")

# Metrics for LR model with C=0.01
auc_c01 = roc_auc_score(y_clf_test, y_proba_c01)
precision_c01 = precision_score(y_clf_test, y_pred_c01)
recall_c01 = recall_score(y_clf_test, y_pred_c01)
f1_c01 = f1_score(y_clf_test, y_pred_c01)

print("\nLogistic Regression (C=0.01 - stronger regularization):")
print(f"  ROC AUC: {auc_c01:.4f}")
print(f"  Precision: {precision_c01:.4f}")
print(f"  Recall: {recall_c01:.4f}")
print(f"  F1-Score: {f1_c01:.4f}")



--- Regularization Experiment (C=0.01) ---

Default Logistic Regression (C=1.0):
  ROC AUC: 0.6879
  Precision: 0.5777
  Recall: 0.8515
  F1-Score: 0.6884

Logistic Regression (C=0.01 - stronger regularization):
  ROC AUC: 0.6920
  Precision: 0.5766
  Recall: 0.8528
  F1-Score: 0.6880


In [24]:

# ------------------------------------------------------------
# 9. Bootstrap CI – with subsampling to reduce memory
# ------------------------------------------------------------
n_bootstrap = 500
auc_diffs = []
np.random.seed(42)

# Optionally subsample test set for bootstrapping (e.g., 1000 rows)
if len(y_clf_test) > 2000:
    subsample_size = 1000
else:
    subsample_size = len(y_clf_test)

# Pre-store indices for speed
test_indices = np.arange(len(y_clf_test))

for _ in range(n_bootstrap):
    idx = np.random.choice(test_indices, size=subsample_size, replace=True)
    y_true_boot = y_clf_test.iloc[idx].values
    y_proba_c1_boot = y_proba[idx]
    y_proba_c01_boot = y_proba_c01[idx]
    auc_c1_boot = roc_auc_score(y_true_boot, y_proba_c1_boot)
    auc_c01_boot = roc_auc_score(y_true_boot, y_proba_c01_boot)
    auc_diffs.append(auc_c1_boot - auc_c01_boot)

mean_diff = np.mean(auc_diffs)
lower = np.percentile(auc_diffs, 2.5)
upper = np.percentile(auc_diffs, 97.5)
print(f"Mean diff: {mean_diff:.4f}, 95% CI: [{lower:.4f}, {upper:.4f}]")

# ------------------------------------------------------------
# Clean up (optional)
# ------------------------------------------------------------
# del X_train_scaled, X_test_scaled, y_reg_train, ... etc.

Mean diff: -0.0040, 95% CI: [-0.0087, -0.0001]
